# Demo 2 — Matplotlib Figure and Axes fundamentals

For each chart, state the question, audience need, intended descriptive claim or comparison, displayed unit, grain, and variable roles before choosing the plotting call.

Colab is the default launch path; the same source runs in clean local Jupyter. The setup cell installs only mismatched course packages before their first import. Colab files are ephemeral, and edits made in a GitHub-opened Colab tab are not automatically saved to GitHub.

This demo uses only course-authored synthetic prepared data. Do not add uploads, Drive mounts, network data, credentials, private records, or sensitive output. Stored notebook output is not execution evidence: restart and run all from a fresh runtime. Assignment Colab submission remains conditional on the repository-save/Classroom50 pilot.


## Figure, Axes, and prepared data

A **Figure** is the complete canvas that can be saved. An **Axes** is one plotting area inside the Figure, including its data region, scales, labels, title, and marks. An x-axis or y-axis is one component of an Axes.

`program_progress.csv` has grain one prepared program-round score per row. `participant_scores.csv` has grain one participant per row. State the variable roles and intended comparison before constructing each chart.


In [ ]:
import importlib.metadata
import platform
from pathlib import Path
import subprocess
import sys

EXPECTED_PACKAGES = {
    "numpy": "2.0.2",
    "pandas": "3.0.3",
    "matplotlib": "3.10.8",
    "seaborn": "0.13.2",
}

assert platform.python_version() == "3.12.13", (
    "Select the course Python 3.12.13 runtime before continuing; found "
    f"{platform.python_version()}."
)

install_specs = []
for package_name, expected_version in EXPECTED_PACKAGES.items():
    try:
        installed_version = importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        installed_version = None
    if installed_version != expected_version:
        install_specs.append(f"{package_name}=={expected_version}")

if install_specs:
    print("Installing mismatched course packages:", ", ".join(install_specs))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *install_specs]
    )

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

actual_versions = {
    "Python": platform.python_version(),
    "NumPy": np.__version__,
    "pandas": pd.__version__,
    "Matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
}
print(actual_versions)
assert actual_versions == {
    "Python": "3.12.13",
    "NumPy": "2.0.2",
    "pandas": "3.0.3",
    "Matplotlib": "3.10.8",
    "seaborn": "0.13.2",
}

BLUE = "#0072B2"
ORANGE = "#D55E00"
GREEN = "#009E73"
PURPLE = "#CC79A7"


In [ ]:
from hashlib import sha256

FIXTURES = {'program_progress.csv': {'text': 'program,round_number,score\nStandard,1,62\nStandard,2,65\nStandard,3,67\nStandard,4,70\nStandard,5,72\nGuided,1,61\nGuided,2,66\nGuided,3,71\nGuided,4,75\nGuided,5,79\n', 'sha256': 'c48d53634f711d4f60b32f230633a47c77e56d8b1eac5f8c84fbad3858f85b36'}, 'participant_scores.csv': {'text': 'participant_id,program,practice_hours,score\nS01,Standard,2.0,61\nS02,Standard,3.0,65\nS03,Standard,3.5,66\nS04,Standard,4.0,67\nS05,Standard,5.0,69\nG01,Guided,2.5,64\nG02,Guided,3.5,70\nG03,Guided,4.0,73\nG04,Guided,5.0,77\nG05,Guided,6.0,82\n', 'sha256': '8eecd1393f3dbd4599269ba41724b28325ea2035f926ffeca674c2150abfc165'}}


def find_demo_directory(start):
    current = start.resolve()
    while True:
        for candidate in (current, current / "07" / "demo"):
            if (
                (candidate / "DEMO_GUIDE.md").is_file()
                and (candidate / ".python-version").is_file()
            ):
                return candidate
        if current.parent == current:
            return None
        current = current.parent


DEMO_DIRECTORY = find_demo_directory(Path.cwd())
if DEMO_DIRECTORY is None:
    DEMO_DIRECTORY = Path.cwd().resolve()

DATA_DIRECTORY = DEMO_DIRECTORY / "data"
DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
FIXTURE_PATHS = {}
for fixture_name, fixture_contract in FIXTURES.items():
    fixture_path = DATA_DIRECTORY / fixture_name
    if not fixture_path.exists():
        fixture_path.write_text(fixture_contract["text"], encoding="utf-8")
    actual_checksum = sha256(fixture_path.read_bytes()).hexdigest()
    assert actual_checksum == fixture_contract["sha256"], (
        f"{fixture_name} does not match the supplied fixture checksum. "
        "Restore the committed file; corrupt data are never replaced silently."
    )
    FIXTURE_PATHS[fixture_name] = fixture_path

OUTPUT_DIRECTORY = DEMO_DIRECTORY / "output"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
for output_name in ['core_line_chart.png']:
    output_path = OUTPUT_DIRECTORY / output_name
    if output_path.exists():
        output_path.unlink()

print("Demo directory:", DEMO_DIRECTORY)
print("Fixtures:", FIXTURE_PATHS)
print("Output directory:", OUTPUT_DIRECTORY)

progress = pd.read_csv(FIXTURE_PATHS["program_progress.csv"])
participants = pd.read_csv(FIXTURE_PATHS["participant_scores.csv"])
assert progress.shape == (10, 3)
assert participants.shape == (10, 4)
assert progress[["program", "round_number"]].duplicated().sum() == 0
assert participants["participant_id"].is_unique
print(progress)
print(participants)


## 1. Line chart — ordered change

Question: how the two prepared score paths change across ordered study rounds. `round_number` is ordered, `score` is quantitative, and `program` is categorical. Each point represents one program-round prepared summary. Color plus marker and line style identify programs redundantly.


In [ ]:
standard_rounds = progress.loc[progress["program"].eq("Standard")]
guided_rounds = progress.loc[progress["program"].eq("Guided")]

line_figure, line_ax = plt.subplots(figsize=(7, 4))
line_ax.plot(
    standard_rounds["round_number"],
    standard_rounds["score"],
    color=BLUE,
    marker="o",
    linestyle="-",
    label="Standard",
)
line_ax.plot(
    guided_rounds["round_number"],
    guided_rounds["score"],
    color=ORANGE,
    marker="s",
    linestyle="--",
    label="Guided",
)
line_ax.set(
    title="Prepared scores across study rounds",
    xlabel="Study round",
    ylabel="Prepared score (points)",
    xticks=[1, 2, 3, 4, 5],
)
line_ax.legend(title="Program")
line_figure.tight_layout()
line_path = OUTPUT_DIRECTORY / "core_line_chart.png"
line_figure.savefig(line_path, dpi=150, bbox_inches="tight")

assert len(line_ax.lines) == 2
assert [item.get_marker() for item in line_ax.lines] == ["o", "s"]
assert [item.get_linestyle() for item in line_ax.lines] == ["-", "--"]
assert line_ax.get_xlabel() == "Study round"
assert line_ax.get_ylabel() == "Prepared score (points)"


## 2. Bar chart — category magnitudes

The two-row table below is already prepared; Lecture 07 does not aggregate it. Bar length encodes a quantitative magnitude across program categories, so the ordinary baseline is zero.


In [ ]:
prepared_program_means = pd.DataFrame(
    {"program": ["Standard", "Guided"], "mean_score": [65.6, 73.2]}
)
bar_figure, bar_ax = plt.subplots(figsize=(6, 4))
bars = bar_ax.bar(
    prepared_program_means["program"],
    prepared_program_means["mean_score"],
    color=[BLUE, ORANGE],
)
bar_ax.set(
    title="Prepared mean score by program",
    xlabel="Program",
    ylabel="Prepared mean score (points)",
    ylim=(0, 80),
)
bar_ax.bar_label(bars, fmt="%.1f")
bar_figure.tight_layout()

assert bar_ax.get_ylim()[0] == 0
assert len(bars) == 2
assert [bar.get_height() for bar in bars] == [65.6, 73.2]


## 3. Scatter plot — two quantitative variables

Each point is one participant. The chart describes how practice hours and observed scores vary in this prepared table. It does not calculate correlation, fit a trend, or establish causation.


In [ ]:
scatter_figure, scatter_ax = plt.subplots(figsize=(6, 4))
for program, color, marker in [
    ("Standard", BLUE, "o"),
    ("Guided", ORANGE, "s"),
]:
    subset = participants.loc[participants["program"].eq(program)]
    scatter_ax.scatter(
        subset["practice_hours"],
        subset["score"],
        color=color,
        marker=marker,
        s=55,
        label=program,
    )
scatter_ax.set(
    title="Practice hours and observed score",
    xlabel="Practice (hours)",
    ylabel="Observed score (points)",
)
scatter_ax.legend(title="Program")
scatter_figure.tight_layout()

assert len(scatter_ax.collections) == 2
assert sum(len(collection.get_offsets()) for collection in scatter_ax.collections) == 10


## 4. Histogram — one quantitative distribution

A bin is a numeric interval represented by one bar. Fixed edges make the displayed score distribution reproducible; the y-axis counts participants.


In [ ]:
histogram_figure, histogram_ax = plt.subplots(figsize=(6, 4))
bin_edges = [60, 65, 70, 75, 80, 85]
counts, returned_edges, histogram_patches = histogram_ax.hist(
    participants["score"],
    bins=bin_edges,
    color=GREEN,
    edgecolor="white",
)
histogram_ax.set(
    title="Distribution of observed participant scores",
    xlabel="Observed score (points)",
    ylabel="Participants (count)",
)
histogram_figure.tight_layout()

assert int(counts.sum()) == len(participants)
assert returned_edges.tolist() == bin_edges
assert len(histogram_patches) == 5


## 5. Box plot — compact distribution summaries

The median is the middle ordered value; Q1 and Q3 bound the central half; IQR is `Q3 - Q1`; and default whiskers extend to observed values within `1.5 × IQR`. Points beyond whiskers are observations to investigate in context, not automatic errors or deletion candidates.


In [ ]:
standard_scores = participants.loc[
    participants["program"].eq("Standard"), "score"
]
guided_scores = participants.loc[
    participants["program"].eq("Guided"), "score"
]

box_figure, box_ax = plt.subplots(figsize=(6, 4))
box_artists = box_ax.boxplot(
    [standard_scores, guided_scores],
    tick_labels=["Standard", "Guided"],
    orientation="vertical",
    whis=1.5,
    patch_artist=True,
)
for patch, color in zip(box_artists["boxes"], [BLUE, ORANGE]):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
box_ax.set(
    title="Observed score distributions by program",
    xlabel="Program",
    ylabel="Observed score (points)",
)
box_figure.tight_layout()

assert len(box_artists["boxes"]) == 2
assert len(box_artists["medians"]) == 2


## Human visual QA

Inspect the newly rendered figures. Confirm that every chart type matches its intended comparison, labels and units are readable, the bar starts at zero, identities do not rely on color alone when needed, the histogram bins are interpretable, and the scatter/box prose stays descriptive.


In [ ]:
assert line_path.is_file() and line_path.stat().st_size > 1_000
line_pixels = plt.imread(line_path)
assert line_pixels.shape[0] > 300 and line_pixels.shape[1] > 500
assert line_ax.get_title() and bar_ax.get_title() and scatter_ax.get_title()
assert histogram_ax.get_ylabel() == "Participants (count)"
assert box_ax.get_ylabel() == "Observed score (points)"
print("Wrote:", line_path)
print("Demo 2 fresh-run verification passed")
